# GenAI-Traces: Anomaly Detection

This notebook demonstrates:
- Statistical anomaly detection using Z-scores
- Detecting cost spikes and latency regressions
- Alert management

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. Anomaly Detection Basics

In [ ]:
from genai_traces.telemetry.anomaly import AnomalyDetector, AnomalyEvent
import random

detector = AnomalyDetector(window=50, z_threshold=3.0)

# Build up baseline with normal values
print("Building baseline with normal cost values...")
for i in range(60):
    # Normal cost around $0.002 with small variation
    cost = random.gauss(0.002, 0.0003)
    detector.observe("gpt-4o", "cost_usd", cost)

baseline = detector.get_baseline("gpt-4o", "cost_usd")
print(f"\nBaseline statistics:")
print(f"  Mean: ${baseline['mean']:.6f}")
print(f"  Stdev: ${baseline['stdev']:.6f}")
print(f"  Samples: {baseline['n']}")

In [ ]:
# Test anomaly detection
print("\nTesting anomaly detection:")
print("=" * 60)

test_values = [
    0.002,   # Normal
    0.0025,  # Slightly high but normal
    0.008,   # Anomaly - 4x normal
    0.015,   # Major anomaly - 7.5x normal
    0.001,   # Normal (low)
]

for value in test_values:
    anomaly = detector.check("gpt-4o", "cost_usd", value)
    if anomaly:
        print(f"\n🚨 ANOMALY DETECTED")
        print(f"   Value: ${value:.6f}")
        print(f"   Baseline: ${anomaly.baseline:.6f}")
        print(f"   Z-score: {anomaly.z_score:.2f}")
        print(f"   Severity: {anomaly.severity}")
    else:
        print(f"✅ Normal: ${value:.6f}")

## 2. Multi-Metric Monitoring

In [ ]:
# Create a fresh detector
detector = AnomalyDetector(window=30, z_threshold=2.5)

# Build baselines for multiple metrics
print("Building baselines for multiple metrics...")

for i in range(40):
    detector.observe("gpt-4o", "cost_usd", random.gauss(0.003, 0.0005))
    detector.observe("gpt-4o", "latency_ms", random.gauss(500, 50))
    detector.observe("gpt-4o", "tokens", random.gauss(1000, 100))

# Print baselines
for metric in ["cost_usd", "latency_ms", "tokens"]:
    baseline = detector.get_baseline("gpt-4o", metric)
    print(f"\n{metric}:")
    print(f"  Mean: {baseline['mean']:.2f}, Stdev: {baseline['stdev']:.2f}")

In [ ]:
# Simulate a problematic request
print("\nSimulating problematic request:")
print("=" * 60)

# Check multiple metrics at once
metrics = {
    "cost_usd": 0.015,    # 5x normal
    "latency_ms": 2000,   # 4x normal
    "tokens": 1100,       # Normal
}

anomalies = []
for metric, value in metrics.items():
    detector.observe("gpt-4o", metric, value)
    anomaly = detector.check("gpt-4o", metric, value)
    if anomaly:
        anomalies.append(anomaly)
        print(f"\n🚨 {anomaly}")
    else:
        print(f"✅ {metric}: {value} (normal)")

print(f"\nTotal anomalies detected: {len(anomalies)}")

## 3. Alert Management

In [ ]:
from genai_traces.telemetry.anomaly import AlertManager

# Track alerts sent
alerts_received = []

def custom_alert_handler(event: AnomalyEvent):
    alerts_received.append(event)
    print(f"📧 Custom alert: {event}")

# Create alert manager with log and custom callback
alert_manager = AlertManager(channels=[
    {"type": "log"},
    {"type": "callback", "fn": custom_alert_handler},
])

# Send alerts for detected anomalies
print("Sending alerts:")
print("=" * 60)

for anomaly in anomalies:
    alert_manager.send(anomaly)

print(f"\nTotal alerts sent: {len(alerts_received)}")

## 4. Integrated Anomaly Detection with Tracing

In [ ]:
from genai_traces import init_tracer
from genai_traces.exporters import ConsoleExporter
from genai_traces.core.types import SpanType
import time

tracer = init_tracer(
    service_name="anomaly-demo",
    exporters=[ConsoleExporter(pretty=True)],
)

# Fresh detector and alert manager
detector = AnomalyDetector(window=20, z_threshold=2.5)
alert_manager = AlertManager(channels=[{"type": "log"}])

def monitored_llm_call(cost: float, latency_ms: float):
    """LLM call with anomaly monitoring."""
    with tracer.start_as_current_span("monitored_llm", SpanType.LLM) as span:
        span.set_attribute("llm.model.name", "gpt-4o")
        span.set_attribute("cost.total_usd", cost)
        span.duration_ms = latency_ms
        
        # Check for anomalies
        anomalies = []
        
        detector.observe("gpt-4o", "cost_usd", cost)
        cost_anomaly = detector.check("gpt-4o", "cost_usd", cost)
        if cost_anomaly:
            anomalies.append(cost_anomaly)
            span.set_attribute("anomaly.cost_detected", True)
            span.set_attribute("anomaly.cost_z_score", cost_anomaly.z_score)
        
        detector.observe("gpt-4o", "latency_ms", latency_ms)
        latency_anomaly = detector.check("gpt-4o", "latency_ms", latency_ms)
        if latency_anomaly:
            anomalies.append(latency_anomaly)
            span.set_attribute("anomaly.latency_detected", True)
            span.set_attribute("anomaly.latency_z_score", latency_anomaly.z_score)
        
        # Send alerts
        for anomaly in anomalies:
            alert_manager.send(anomaly)
        
        return len(anomalies)

# Build baseline
print("Building baseline (20 normal calls)...")
for i in range(20):
    monitored_llm_call(
        cost=random.gauss(0.003, 0.0003),
        latency_ms=random.gauss(400, 30)
    )

print("\nBaseline established.")
print("\n" + "=" * 60)
print("Simulating anomalous call:")
print("=" * 60)

# Trigger anomaly
anomaly_count = monitored_llm_call(cost=0.02, latency_ms=1500)
print(f"\nAnomalies detected in call: {anomaly_count}")

## Summary

This notebook demonstrated:
- ✅ Statistical anomaly detection with Z-scores
- ✅ Building baselines from observations
- ✅ Multi-metric monitoring
- ✅ Severity classification
- ✅ Alert management with multiple channels
- ✅ Custom alert handlers
- ✅ Integrated anomaly detection with tracing